# Calibrating the advice constants

The advice view (Spend / Save tags in Let It Ride) values a golfer's **future** windows with two judgment calls in `scripts/predictions/advice_values.py`:

| Constant | Current | Meaning |
|---|---|---|
| `P_PLAYS_OPEN` | 0.60 | chance he enters a later *open* event |
| `P_PLAYS_RESTRICTED` | 0.90 | chance he enters a later major / signature / playoff he qualifies for |
| `DECAY_PER_WEEK` | 0.98 | how fast "what he's worth now" stops predicting "what he'll be worth then" |

This notebook measures them from the 2026 season instead of guessing.

**Data:** `data/analysis/advice_calibration_pairs.csv` — one row per (golfer, a week we predicted, a later event). Rebuild it any time with `python3 scripts/analysis/build_advice_calibration.py`.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/analysis/advice_calibration_pairs.csv")
print(f"{len(df):,} pairs")
df.head()

## 1. How often does a golfer actually play a later event?

`played_b` is True when he teed it up at event B.

**Task:** compute the play rate split by `restricted_b` (open vs restricted events).

Then split restricted events by whether he's likely *qualified*. We don't have the qualification list here, so use world rank as the proxy the advice uses: `world_rank <= 60`.

Hint: `df.groupby(...)["played_b"].mean()` — and look at the counts too (`.agg(["mean", "count"])`) so you know how much to trust each number.

In [ ]:
# TODO: play rate for open vs restricted events


In [ ]:
# TODO: restricted events only — play rate for world_rank <= 60 vs > 60


**Think about it:** does the play rate depend on how good the golfer is, even at open events? A world #5 skips most opposite-field events; a #120 plays nearly everything he can get into. Try bucketing `world_rank` (e.g. `pd.cut(df.world_rank, [0, 10, 30, 60, 120, 1000])`) and see whether one constant is enough or the advice should use a few.

In [ ]:
# TODO: play rate by world-rank bucket × restricted_b


## 2. How fast does today's value go stale?

When a golfer *did* play B and B had a Tuesday prediction (`ev_b_model` not null), compare:

- `ev_extrap` — what the advice would have predicted for him at B from week A (purse + field-strength adjusted, **before** decay)
- `ev_b_model` — what the model actually said about him on B's Tuesday

If form were permanent, `ev_b_model / ev_extrap` would average ~1.0 at every distance. Decay means the ratio drifts as `weeks_apart` grows.

**Task:** filter to rows with both numbers, compute the ratio, and look at its **median** by `weeks_apart` (rounded to whole weeks). Median, not mean — a few big ratios from long shots will skew a mean.

In [ ]:
# TODO: ratio = ev_b_model / ev_extrap, median by round(weeks_apart)


**Fit the decay:** if value decays by a constant factor `d` per week, then `median_ratio ≈ d ** weeks`, so `log(median_ratio) ≈ weeks × log(d)`. A straight-line fit through the origin of `log(ratio)` against `weeks` gives `log(d)`.

Hint: `slope = (w * y).sum() / (w * w).sum()` is the least-squares slope through the origin for x=`w`, y=`y`. Then `d = np.exp(slope)`.

In [ ]:
# TODO: fit d, then compare with the current 0.98


## 3. Sanity check against real money (optional)

`earnings_b` is what he actually earned at B. For golfers who played, compare the mean of `earnings_b` with the mean of `ev_extrap × your decay` by weeks_apart. Golf payouts are lumpy (one win dwarfs everything), so expect noise — this is a smell test, not the fit.

## 4. Write down what you found

Fill in the numbers you'd put in `advice_values.py` and why. Then we'll change the constants, rerun Bank of Utah's advice, and see how the Spend / Save split moves.

| Constant | Old | New | Evidence (n) |
|---|---|---|---|
| P_PLAYS_OPEN | 0.60 | | |
| P_PLAYS_RESTRICTED | 0.90 | | |
| DECAY_PER_WEEK | 0.98 | | |
